# Ensemble Model-1-3-4 (Individual characteristics + breath-by-breath + beat-by-beat)

Use the Python 3.11 kernel and private preprocessed data; see [setup](../README.md). Run cells top to bottom.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from ai_cpet_ots.dataloaders import (
    Model_1_3_4_dataset,
    create_dataset_model_3,
    create_dataset_model_4,
    load_clinical_data,
)
from ai_cpet_ots.helpers import collate_model_1_3_4
from ai_cpet_ots.models import Model_1, Model_1_3_4, Model_3, Model_4
from ai_cpet_ots.paths import (
    BBB_CONTROL_DIR,
    BBB_OTS_DIR,
    CLIN_CONTROL_DIR,
    CLIN_OTS_DIR,
    ECG_CONTROL_DIR,
    ECG_OTS_DIR,
    MODEL_WEIGHTS,
    output_path,
)
from ai_cpet_ots.testers import test_model_1_3_4

In [ ]:
def optimize_model(
    X_train_clinical, Y_train_clinical, X_train_bbb, Y_train_bbb, X_train_ecg, Y_train_ecg
):
    """Search the notebook's study hyperparameter bounds using training inputs only.

    Returns the optimizer result used by train_model. Requires the optimization
    extra and can be expensive; it is not run in the default evaluation workflow."""
    from ai_cpet_ots.optimizers import Optimizer_Model_1_3_4

    # Optimize model
    opt = Optimizer_Model_1_3_4(
        model_name="Model_1_3_4",
        X_train_clinical=X_train_clinical,
        Y_train_clinical=Y_train_clinical,
        X_train_bbb=X_train_bbb,
        Y_train_bbb=Y_train_bbb,
        X_train_ecg=X_train_ecg,
        Y_train_ecg=Y_train_ecg,
    )

    bounds = [
        {"name": "n_epochs", "type": "discrete", "domain": [5, 5.000001]},
        {"name": "lr", "type": "discrete", "domain": (1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1)},
        {
            "name": "weight_decay",
            "type": "discrete",
            "domain": (0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1),
        },
        {"name": "batch_size", "type": "discrete", "domain": [64, 128]},
    ]

    _, opt_params_gpbo = opt.run_bo_gp(bounds, 10)

    return opt_params_gpbo

In [ ]:
def train_model(
    opt_params,
    X_train_clinical,
    Y_train_clinical,
    X_train_bbb,
    Y_train_bbb,
    X_train_ecg,
    Y_train_ecg,
):
    """Train this notebook's model from opt_params and the supplied training arrays.

    Keeps the original objective, optimizer, and epoch settings. Returns the
    trained model; checkpoint saving is controlled separately by save_model."""
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
    else:
        device = torch.device("cpu")

    n_epochs, lr, weight_decay, batch_size, threshold = opt_params

    # Shuffle trainset and create minibatch
    trainset = Model_1_3_4_dataset(
        X_train_clinical, Y_train_clinical, X_train_bbb, Y_train_bbb, X_train_ecg, Y_train_ecg
    )
    trainloader = DataLoader(
        trainset, batch_size=batch_size, shuffle=True, collate_fn=collate_model_1_3_4
    )

    # Initializing models
    model_1 = Model_1().to(device)
    model_1.load_state_dict(
        torch.load(MODEL_WEIGHTS / "Model_1.pth", map_location=device, weights_only=True)
    )
    model_1.eval()

    model_3 = Model_3().to(device)
    model_3.load_state_dict(
        torch.load(MODEL_WEIGHTS / "Model_3.pth", map_location=device, weights_only=True)
    )
    model_3.eval()

    model_4 = Model_4().to(device)
    model_4.load_state_dict(
        torch.load(MODEL_WEIGHTS / "Model_4.pth", map_location=device, weights_only=True)
    )
    model_4.eval()

    model = Model_1_3_4(threshold).to(device)

    pos_samples = np.sum(Y_train_clinical)
    neg_samples = len(Y_train_clinical) - pos_samples
    pos_weight = torch.tensor([neg_samples / pos_samples]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Training
    for e in range(n_epochs):
        train_loss = 0
        for batch in trainloader:
            x_batch_clinical, y_batch, x_batch_bbb, _, x_batch_ecg, _, _ = batch
            x_batch_clinical, y_batch, x_batch_bbb, x_batch_ecg = (
                x_batch_clinical.float().to(device),
                y_batch.to(device),
                x_batch_bbb.float().to(device),
                x_batch_ecg.float().to(device),
            )

            # Clinical & beat-by-beat
            with torch.no_grad():
                pred_clin, _ = model_1(x_batch_clinical)
                pred_ecg, _ = model_4(x_batch_ecg)

            x_splitted = x_batch_bbb.unfold(dimension=1, size=30, step=15).to(device)
            preds_bbb = torch.empty((0)).to(device)

            for i in range(x_splitted.size(1)):
                win = x_splitted[:, i, :, :]  # (batch, feats, win_len)

                with torch.no_grad():
                    pred_bbb, _, _, _ = model_3(win)

            preds_bbb = torch.cat((preds_bbb, pred_bbb.unsqueeze(0)), dim=0)

            pred_bbb = preds_bbb.mean(0).flatten()

            y_pred = model(
                pred_clin.flatten(), pred_bbb, pred_ecg.flatten()
            )  # Weighted average of predictions

            loss = criterion(y_pred, y_batch.float())
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss.item() / len(trainloader)

        print(f"Epoch [{e + 1}/{n_epochs}] - Loss {train_loss:.4f}")

    return model

In [ ]:
def test_model(
    X_test_clinical,
    Y_test_clinical,
    X_test_bbb,
    Y_test_bbb,
    X_test_ecg,
    Y_test_ecg,
    model=None,
    save_fig=False,
):
    """Load the published checkpoint when model is None and evaluate test inputs.

    Constructs the study DataLoader and forwards plot options to the reusable
    evaluator. Private inputs must use the original feature order and scaling."""
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
    else:
        device = torch.device("cpu")

    model_1 = Model_1().to(device)
    model_1.load_state_dict(
        torch.load(MODEL_WEIGHTS / "Model_1.pth", map_location=device, weights_only=True)
    )
    model_1.eval()

    model_3 = Model_3().to(device)
    model_3.load_state_dict(
        torch.load(MODEL_WEIGHTS / "Model_3.pth", map_location=device, weights_only=True)
    )
    model_3.eval()

    model_4 = Model_4().to(device)
    model_4.load_state_dict(
        torch.load(MODEL_WEIGHTS / "Model_4.pth", map_location=device, weights_only=True)
    )
    model_4.eval()

    if model is None:
        model = Model_1_3_4().to(device)
        model.load_state_dict(
            torch.load(MODEL_WEIGHTS / "Model_1_3_4.pth", map_location=device, weights_only=True)
        )

    # Test model
    testset = Model_1_3_4_dataset(
        X_test_clinical, Y_test_clinical, X_test_bbb, Y_test_bbb, X_test_ecg, Y_test_ecg
    )
    testloader = DataLoader(
        testset, batch_size=len(testset), shuffle=False, collate_fn=collate_model_1_3_4
    )

    auroc, auprc, f1, acc = test_model_1_3_4(
        model,
        model_1,
        model_3,
        model_4,
        "Model 1-3-4",
        testloader,
        plot_cm=True,
        plot_PRC=True,
        plot_predictions=True,
        plot_ROC=True,
        save_fig=save_fig,
    )
    print(f"AUCPR: {auprc:.3f}, F1: {f1:.3f}, ACC: {acc:.3f}, Threshold {model.threshold:.3f}")

In [ ]:
training_model = False
save_model = False
save_fig = False
model_name = "Model_1_3_4"

In [ ]:
"""Load clinical data for model 1"""
_, _, X_train_clinical, Y_train_clinical, X_test_clinical, Y_test_clinical = load_clinical_data(
    CLIN_CONTROL_DIR, CLIN_OTS_DIR
)

In [ ]:
"""Load BBB data for model 3"""
_, _, X_train_bbb, Y_train_bbb, X_test_bbb, Y_test_bbb, _, variables = create_dataset_model_3(
    BBB_CONTROL_DIR, BBB_OTS_DIR, mode="minmax"
)

In [ ]:
"""Load ECG data for model 4"""
X_train_ecg, Y_train_ecg, X_test_ecg, Y_test_ecg = create_dataset_model_4(
    ECG_CONTROL_DIR, ECG_OTS_DIR, filter_mode="pan-tompkins"
)

In [ ]:
if training_model:
    # Optimize model
    print("Optimizing model...")
    opt_params = optimize_model(
        X_train_clinical, Y_train_clinical, X_train_bbb, Y_train_bbb, X_train_ecg, Y_train_ecg
    )

    # Train the model
    print("Training model...")
    model = train_model(
        opt_params,
        X_train_clinical,
        Y_train_clinical,
        X_train_bbb,
        Y_train_bbb,
        X_train_ecg,
        Y_train_ecg,
    )

    # Test model
    print("Testing model...")
    test_model(
        X_test_clinical, Y_test_clinical, X_test_bbb, Y_test_bbb, X_test_ecg, Y_test_ecg, model
    )

    if save_model:
        torch.save(model.state_dict(), output_path(f"{model_name}.pth"))
else:
    print("Testing model...")
    test_model(
        X_test_clinical,
        Y_test_clinical,
        X_test_bbb,
        Y_test_bbb,
        X_test_ecg,
        Y_test_ecg,
        save_fig=save_fig,
    )